# Logistic Regression Experiments
This notebook is a lightweight sandbox for primitive test runs.

Run cells in order from top to bottom.

In [6]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Data root in this repo
BASE_DIR = "datasets_full.csv"
USERS_FILE = "users.csv"

DATASETS = {
    "genuine_accounts.csv": 0,
    "social_spambots_1.csv": 1,
    "social_spambots_2.csv": 1,
    "social_spambots_3.csv": 1,
    "traditional_spambots_1.csv": 1,
    "traditional_spambots_2.csv": 1,
    "traditional_spambots_3.csv": 1,
    "traditional_spambots_4.csv": 1,
}

NUMERIC_FEATURES = [
    "statuses_count", "followers_count", "friends_count",
    "favourites_count", "listed_count", "default_profile",
    "default_profile_image", "verified"
]

TEXT_FEATURE = "description"

print("Dataset root:", BASE_DIR)
print("Files configured:", len(DATASETS))

Dataset root: datasets_full.csv
Files configured: 8


In [7]:
def resolve_users_csv(base_dir, dataset_entry):
    """Resolve dataset entry to a users.csv path."""
    candidates = [
        os.path.join(base_dir, dataset_entry),
        os.path.join(base_dir, dataset_entry, USERS_FILE),
        os.path.join(base_dir, dataset_entry, dataset_entry, USERS_FILE),
    ]

    for p in candidates:
        if os.path.isfile(p):
            return p
    return None


def load_all_data(dataset_map, base_dir):
    """Load all configured datasets and attach binary labels."""
    frames = []
    missing_files = []

    for dataset_entry, label in dataset_map.items():
        path = resolve_users_csv(base_dir, dataset_entry)
        if path is None:
            missing_files.append(dataset_entry)
            continue

        df = pd.read_csv(
            path,
            encoding="utf-8",
            on_bad_lines="skip",
            low_memory=False,
        )
        df["label"] = int(label)
        df["source_file"] = dataset_entry
        frames.append(df)
        print(f"Loaded {dataset_entry}: {len(df):,} rows")

    if missing_files:
        print("\nMissing dataset entries:")
        for p in missing_files:
            print("-", p)

    if not frames:
        raise ValueError("No dataset files were loaded. Check BASE_DIR and DATASETS.")

    all_data = pd.concat(frames, ignore_index=True)
    print(f"\nTotal rows loaded: {len(all_data):,}")
    return all_data

# Block 1: Data loading (full configured datasets)
raw = load_all_data(DATASETS, BASE_DIR)
raw.shape

Loaded genuine_accounts.csv: 3,474 rows
Loaded social_spambots_1.csv: 991 rows
Loaded social_spambots_2.csv: 3,457 rows
Loaded social_spambots_3.csv: 464 rows
Loaded traditional_spambots_1.csv: 1,000 rows
Loaded traditional_spambots_2.csv: 100 rows
Loaded traditional_spambots_3.csv: 403 rows
Loaded traditional_spambots_4.csv: 1,128 rows

Total rows loaded: 11,017


(11017, 44)

In [8]:
# Block 2: Slicing and preprocessing
available_numeric = [c for c in NUMERIC_FEATURES if c in raw.columns]
if not available_numeric:
    raise ValueError("No numeric feature columns found in raw data.")

X = raw[available_numeric].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

if TEXT_FEATURE in raw.columns:
    X["description_len"] = raw[TEXT_FEATURE].fillna("").astype(str).str.len()
else:
    X["description_len"] = 0

y = raw["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class balance:\n", y.value_counts(dropna=False))

Train shape: (8813, 9)
Test shape: (2204, 9)
Class balance:
 label
1    7543
0    3474
Name: count, dtype: int64


In [9]:
# Block 3: Training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_scaled, y_train)

print("Training complete.")
print("Model intercept:", model.intercept_[0])

Training complete.
Model intercept: -2.8517249546277483


In [10]:
# Block 4: Evaluation
pred = model.predict(X_test_scaled)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0],
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 10 features by |coefficient|:")
display(coef_df.head(10))

Accuracy: 0.8961

Confusion Matrix:
[[ 558  137]
 [  92 1417]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8585    0.8029    0.8297       695
           1     0.9118    0.9390    0.9252      1509

    accuracy                         0.8961      2204
   macro avg     0.8852    0.8710    0.8775      2204
weighted avg     0.8950    0.8961    0.8951      2204


Top 10 features by |coefficient|:


,feature,coefficient,abs_coefficient
3,favourites_count,-16.844263,16.844263
2,friends_count,0.959591,0.959591
0,statuses_count,-0.942773,0.942773
4,listed_count,-0.689893,0.689893
5,default_profile,-0.642752,0.642752
8,description_len,-0.413956,0.413956
7,verified,-0.401326,0.401326
1,followers_count,0.355720,0.355720
6,default_profile_image,0.030135,0.030135


## Run Order
1. Cell 2: imports and constants.
2. Cell 3: full data loading.
3. Cell 4: slicing and preprocessing.
4. Cell 5: training.
5. Cell 6: evaluation.

## Notes
- This notebook now loads all configured dataset files directly.
- If runtime or memory gets heavy, reduce loaded columns or process in chunks.